In [ ]:
### XGBoost
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

FRIDAY_PATH = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
WEDNESDAY_PATH = "Wednesday-workingHours.pcap_ISCX.csv"

df_fri = pd.read_csv(FRIDAY_PATH)
df_wed = pd.read_csv(WEDNESDAY_PATH)

data = pd.concat([df_fri, df_wed], ignore_index=True)
print(f"Combined shape: {data.shape}")

data.columns = data.columns.str.strip()
if 'Label' not in data.columns:
    raise ValueError("Expected 'Label' column not found.")

data['Label'] = data['Label'].astype(str).str.strip()
data['Label'] = data['Label'].apply(lambda x: 0 if x == "BENIGN" else 1)

print("\nBinary label distribution (0=BENIGN, 1=ATTACK):")
print(data['Label'].value_counts())

data = data.apply(pd.to_numeric, errors='coerce')
data.replace([np.inf, -np.inf], np.nan, inplace=True)

before = data.shape[0]
data.dropna(inplace=True)
after = data.shape[0]

print(f"\nDropped {before - after} rows due to NaN/inf values.")
print(f"Remaining rows: {after}")

zero_var_cols = [c for c in data.columns if c != 'Label' and data[c].nunique() <= 1]
if zero_var_cols:
    print("Dropping zero-variance columns:", zero_var_cols)
    data.drop(columns=zero_var_cols, inplace=True)

feature_names = [c for c in data.columns if c != 'Label']
X = data[feature_names].values
y = data['Label'].values

print(f"\nFinal feature count: {len(feature_names)}")
print(f"Final dataset size  : {data.shape[0]} rows")

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)

##minmaxscaler
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_trainval = scaler.fit_transform(X_trainval)  
X_test     = scaler.transform(X_test)          
##

xgb_base = XGBClassifier(
    objective="binary:logistic",
    random_state=SEED,
    n_jobs=-1,
    eval_metric="logloss"
)

param_grid = [
    { 
        "n_estimators": [300],
        "max_depth": [6],
        "learning_rate": [0.1],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
    },
    {
        "n_estimators": [400],
        "max_depth": [8],
        "learning_rate": [0.1],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
    },
    {
        "n_estimators": [200],
        "max_depth": [4],
        "learning_rate": [0.1],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
    },
    {
        "n_estimators": [500],
        "max_depth": [6],
        "learning_rate": [0.05],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
    },
    {
        "n_estimators": [300],
        "max_depth": [6],
        "learning_rate": [0.1],
        "subsample": [0.8],
        "colsample_bytree": [0.8],
    },
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True,
    return_train_score=False
)

print("\n CV hyperparameter optimization")
grid.fit(X_trainval, y_trainval)

print("\n Best XGBoost configuration selected by CV ")
print("Best CV accuracy:", f"{grid.best_score_:.6f}")
print("Best params:", grid.best_params_)

xgb_clf = grid.best_estimator_

y_pred_xgb = xgb_clf.predict(X_test)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb, zero_division=0)
rec_xgb = recall_score(y_test, y_pred_xgb, zero_division=0)
f1_xgb = f1_score(y_test, y_pred_xgb, zero_division=0)

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
tn, fp, fn, tp = cm_xgb.ravel()
far_xgb = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print("\n XGBoost Results")
print(f"Accuracy : {acc_xgb:.4f}")
print(f"Precision: {prec_xgb:.4f}")
print(f"Recall   : {rec_xgb:.4f}")
print(f"F1-score : {f1_xgb:.4f}")
print(f"FAR      : {far_xgb:.6f}")

plt.figure(figsize=(5, 4))
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Blues", xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
### Tabnet
import numpy as np
import torch
from copy import deepcopy
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
import os, random
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
print(f"\nShapes -> TrainVal: {X_trainval.shape}, Test: {X_test.shape}")

scaler = MinMaxScaler()

X_trainval = scaler.fit_transform(X_trainval) 
X_test     = scaler.transform(X_test)          

base_params = {
    "n_d": 16,
    "n_a": 16,
    "n_steps": 5,
    "gamma": 1.5,
    "n_independent": 2,
    "n_shared": 2,
    "momentum": 0.02,
    "mask_type": "entmax",
    "lambda_sparse": 1e-4,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": dict(lr=2e-3),
    "scheduler_params": {"step_size": 20, "gamma": 0.95},
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "verbose": 0,
}

param_grid = [
    {
        "name": "first",
        "n_d": 16, "n_a": 16, "n_steps": 5,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "lr": 2e-3,
        "mask_type": "entmax",
    },
    {
        "name": "second",
        "n_d": 16, "n_a": 16, "n_steps": 3,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "lr": 2e-3,
        "mask_type": "entmax",
    },
    {
        "name": "third",
        "n_d": 16, "n_a": 16, "n_steps": 5,
        "gamma": 1.5,
        "lambda_sparse": 1e-3,
        "lr": 2e-3,
        "mask_type": "entmax",
    },
    {
        "name": "fourth",
        "n_d": 16, "n_a": 16, "n_steps": 5,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "lr": 2e-3,
        "mask_type": "sparsemax",
    },
    {
        "name": "fifth",
        "n_d": 16, "n_a": 16, "n_steps": 5,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "lr": 3e-3,
        "mask_type": "entmax",
    },
]


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def build_params(pg):
    params = deepcopy(base_params)
    params["n_d"] = pg["n_d"]
    params["n_a"] = pg["n_a"]
    params["n_steps"] = pg["n_steps"]
    params["gamma"] = pg["gamma"]
    params["lambda_sparse"] = pg["lambda_sparse"]
    params["mask_type"] = pg["mask_type"]
    params["optimizer_params"] = dict(lr=pg["lr"])
    return params

cv_results = []

best_mean = -1.0
best_std = None
best_name = None
best_params = None

print("\n hyperparameter optimization")

for cfg_i, pg in enumerate(param_grid, 1):
    params = build_params(pg)
    fold_scores = []

    print(f"\n-Config {cfg_i}/{len(param_grid)}: {pg['name']}-")

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval), 1):
        X_tr, y_tr = X_trainval[tr_idx], y_trainval[tr_idx]
        X_va, y_va = X_trainval[va_idx], y_trainval[va_idx]

        clf = TabNetClassifier(**params)
        clf.fit(
            X_train=X_tr, y_train=y_tr,
            eval_set=[(X_tr, y_tr), (X_va, y_va)],
            eval_name=["train", "valid"],
            eval_metric=["accuracy"],
            max_epochs=80,
            patience=10,
            batch_size=1024,
            virtual_batch_size=128,
            num_workers=0,
            drop_last=False,
        )

        best_fold_acc = float(np.max(clf.history["valid_accuracy"]))
        fold_scores.append(best_fold_acc)
        print(f"  Fold {fold}: best valid acc = {best_fold_acc:.6f}")

    mean_acc = float(np.mean(fold_scores))
    std_acc = float(np.std(fold_scores))
    
    cv_results.append({
        "name": pg["name"],
        "mean_test_score": mean_acc,
        "std_test_score": std_acc,
        "params": {k: v for k, v in pg.items() if k != "name"},
    })
    print(f"  -> CV mean ± std: {mean_acc:.6f} ± {std_acc:.6f}")

    if mean_acc > best_mean:
        best_mean = mean_acc
        best_std = std_acc
        best_name = pg["name"]
        best_params = params

print("Best CV accuracy:", f"{best_mean:.6f}")
print("Best params:", best_params)

X_tr2, X_va2, y_tr2, y_va2 = train_test_split(
    X_trainval, y_trainval, test_size=0.15, random_state=SEED, stratify=y_trainval
)

tabnet_clf = TabNetClassifier(**best_params)
tabnet_clf.fit(
    X_train=X_tr2, y_train=y_tr2,
    eval_set=[(X_tr2, y_tr2), (X_va2, y_va2)],
    eval_name=["train", "valid"],
    eval_metric=["accuracy"],
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False,
)

y_pred = tabnet_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
far = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"FAR      : {far:.6f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
import shap

def get_outcome_indices(y_true, y_pred):
    """Return dict of indices for TP/FP/FN/TN."""
    return {
        "TP": np.where((y_true == 1) & (y_pred == 1))[0],
        "FP": np.where((y_true == 0) & (y_pred == 1))[0],
        "FN": np.where((y_true == 1) & (y_pred == 0))[0],
        "TN": np.where((y_true == 0) & (y_pred == 0))[0],
    }

def topk_idx(vec, k=10, use_abs=False):
    """Return indices of top-k entries."""
    if use_abs:
        vec = np.abs(vec)
    return np.argsort(vec)[-k:]

def topk_frequency(indices_list, feature_names):
    """Count feature recurrence across top-k sets."""
    if len(indices_list) == 0:
        return pd.DataFrame(columns=["Feature", "Count", "Percent"])
    flat = np.concatenate(indices_list)
    counts = Counter(flat)
    total = len(indices_list)
    rows = []
    for idx, c in counts.most_common():
        rows.append({
            "Feature": feature_names[idx],
            "Count": c,
            "Percent": c / total
        })
    return pd.DataFrame(rows)

def mean_jaccard(topk_sets):
    """Mean Jaccard similarity across pairs of sets."""
    if len(topk_sets) < 2:
        return np.nan
    sims = []
    for a, b in combinations(topk_sets, 2):
        inter = len(a & b)
        union = len(a | b)
        sims.append(inter / union if union else 0.0)
    return float(np.mean(sims)) if sims else np.nan

def sample_indices(arr, n, seed=42):
    """Deterministic sampling without replacement."""
    arr = np.array(arr)
    if len(arr) <= n:
        return arr
    rng = np.random.RandomState(seed)
    return rng.choice(arr, size=n, replace=False)

def plot_top10(df, value_col, title):
    """Simple top-10 bar plot."""
    top10 = df.head(10).copy()
    if top10.empty:
        print("[plot_top10] Empty df, skipping plot:", title)
        return
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top10, x=value_col, y="Feature")
    plt.title(title)
    plt.tight_layout()
    plt.show()

explainer_xgb = shap.TreeExplainer(xgb_clf)
shap_values_xgb = explainer_xgb.shap_values(X_test)

xgb_mean_abs = np.mean(np.abs(shap_values_xgb), axis=0)
df_xgb_global = pd.DataFrame({
    "Feature": feature_names,
    "XGB_MeanAbsSHAP": xgb_mean_abs
}).sort_values("XGB_MeanAbsSHAP", ascending=False)

print("\n=== XGBoost Global SHAP (Top 10) ===")
print(df_xgb_global.head(10).to_string(index=False))
plot_top10(df_xgb_global, "XGB_MeanAbsSHAP", "XGBoost – Global SHAP Feature Importance (Top 10)")

M_tab, masks_tab = tabnet_clf.explain(X_test)

tab_mean_mask = np.mean(M_tab, axis=0)
df_tab_global = pd.DataFrame({
    "Feature": feature_names,
    "TabNet_MeanMask": tab_mean_mask
}).sort_values("TabNet_MeanMask", ascending=False)

print("\n=== TabNet Global Masks (Top 10) ===")
print(df_tab_global.head(10).to_string(index=False))
plot_top10(df_tab_global, "TabNet_MeanMask", "TabNet – Global Intrinsic Feature Importance (Top 10)")

df_global_compare = pd.concat([
    df_xgb_global.head(10).reset_index(drop=True),
    df_tab_global.head(10).reset_index(drop=True)
], axis=1)

print("\n=== Global Top-10 Comparison (XGBoost vs TabNet) ===")
print(df_global_compare.to_string(index=False))

idx_xgb = get_outcome_indices(y_test, y_pred_xgb)
idx_tab = get_outcome_indices(y_test, y_pred) 

print("\n=== Error counts on test set ===")
print(f"XGBoost: FP={len(idx_xgb['FP'])}, FN={len(idx_xgb['FN'])}, TP={len(idx_xgb['TP'])}, TN={len(idx_xgb['TN'])}")
print(f"TabNet : FP={len(idx_tab['FP'])}, FN={len(idx_tab['FN'])}, TP={len(idx_tab['TP'])}, TN={len(idx_tab['TN'])}")

K = 10 

def local_explanation_table_xgb(i):
    """Top-k SHAP features (signed + abs) for one instance."""
    sv = shap_values_xgb[i]
    topk = topk_idx(sv, k=K, use_abs=True)[::-1]
    rows = []
    for j in topk:
        rows.append({
            "Feature": feature_names[j],
            "Value": float(X_test[i, j]),
            "SHAP_Signed": float(sv[j]),
            "SHAP_Abs": float(abs(sv[j]))
        })
    return pd.DataFrame(rows)

def local_explanation_table_tabnet(i):
    """Top-k mask features for one instance."""
    mv = M_tab[i]
    topk = topk_idx(mv, k=K, use_abs=False)[::-1]
    rows = []
    for j in topk:
        rows.append({
            "Feature": feature_names[j],
            "Value": float(X_test[i, j]),
            "MaskWeight": float(mv[j])
        })
    return pd.DataFrame(rows)

xgb_fp_tables = {i: local_explanation_table_xgb(i) for i in idx_xgb["FP"]}
xgb_fn_tables = {i: local_explanation_table_xgb(i) for i in idx_xgb["FN"]}

print("\n=== XGBoost: Example FP explanation (top-k) ===")
if len(idx_xgb["FP"]) > 0:
    print(xgb_fp_tables[idx_xgb["FP"][0]].to_string(index=False))

print("\n=== XGBoost: FN explanation(s) (top-k) ===")
for i in idx_xgb["FN"]:
    print(f"\n-- FN index {i} --")
    print(xgb_fn_tables[i].to_string(index=False))

TAB_FP_N = 30
TAB_FN_N = 30
TAB_TP_N = 30

tab_fp_sample = sample_indices(idx_tab["FP"], TAB_FP_N, seed=SEED)
tab_fn_sample = sample_indices(idx_tab["FN"], TAB_FN_N, seed=SEED)
tab_tp_sample = sample_indices(idx_tab["TP"], TAB_TP_N, seed=SEED)

tab_fp_tables = {i: local_explanation_table_tabnet(i) for i in tab_fp_sample}
tab_fn_tables = {i: local_explanation_table_tabnet(i) for i in tab_fn_sample}

print("\n=== TabNet: FP explanation  ===")
if len(tab_fp_sample) > 0:
    print(tab_fp_tables[int(tab_fp_sample[0])].to_string(index=False))

print("\n=== TabNet: FN explanation ===")
if len(tab_fn_sample) > 0:
    print(tab_fn_tables[int(tab_fn_sample[0])].to_string(index=False))
    
def topk_sets_xgb(indices):
    sets_ = []
    topk_list = []
    for i in indices:
        sv = np.abs(shap_values_xgb[i])
        tk = topk_idx(sv, k=K, use_abs=False)
        topk_list.append(tk)
        sets_.append(set(tk.tolist()))
    return topk_list, sets_

xgb_fp = idx_xgb["FP"]
xgb_tp = sample_indices(idx_xgb["TP"], len(xgb_fp), seed=SEED) if len(xgb_fp) > 0 else idx_xgb["TP"]
xgb_fp_topk, xgb_fp_sets = topk_sets_xgb(xgb_fp)
xgb_tp_topk, xgb_tp_sets = topk_sets_xgb(xgb_tp)
df_xgb_fp_freq = topk_frequency(xgb_fp_topk, feature_names)
df_xgb_tp_freq = topk_frequency(xgb_tp_topk, feature_names)

print("\n XGBoost FP")
print(df_xgb_fp_freq.head(10).to_string(index=False))
print("Jaccard (FP):", mean_jaccard(xgb_fp_sets))

print("\n XGBoost TP")
print(df_xgb_tp_freq.head(10).to_string(index=False))
print("Jaccard (TP):", mean_jaccard(xgb_tp_sets))

print(f"\n[XGBoost] FN count = {len(idx_xgb['FN'])} -> skipping FN consistency metrics (case-study only).")
def topk_sets_tab(indices):
    sets_ = []
    topk_list = []
    for i in indices:
        mv = M_tab[i]
        tk = topk_idx(mv, k=K, use_abs=False)
        topk_list.append(tk)
        sets_.append(set(tk.tolist()))
    return topk_list, sets_

tab_fp_topk, tab_fp_sets = topk_sets_tab(tab_fp_sample)
tab_tp_topk, tab_tp_sets = topk_sets_tab(tab_tp_sample)
df_tab_fp_freq = topk_frequency(tab_fp_topk, feature_names)
df_tab_tp_freq = topk_frequency(tab_tp_topk, feature_names)

print("\nTabnet FP ")
print(df_tab_fp_freq.head(10).to_string(index=False))
print("FP JAccard:", mean_jaccard(tab_fp_sets))

print("\n Tabnet TP")
print(df_tab_tp_freq.head(10).to_string(index=False))
print("TP jaccard:", mean_jaccard(tab_tp_sets))

if len(tab_fn_sample) >= 10:
    tab_fn_topk, tab_fn_sets = topk_sets_tab(tab_fn_sample)
    df_tab_fn_freq = topk_frequency(tab_fn_topk, feature_names)
    print("\n Tabnet FN")
    print(df_tab_fn_freq.head(10).to_string(index=False))
    print("jaccard (FN):", mean_jaccard(tab_fn_sets))
else:
    print(f"\n[TabNet] FN sample size = {len(tab_fn_sample)} -> FN consistency metrics optional / may be unstable.")

